# 🌾 Wheat Plant Disease: Clean Dataset Extraction & Analysis
**Source Pipeline:** [`ssahildhin/wheat-disease-detection`](https://www.kaggle.com/code/ssahildhin/wheat-disease-detection)
**Objective:** Extract the clean dataset by eliminating data leakage and contradictory label conflicts, followed by comprehensive Exploratory Data Analysis (EDA).

---
### 📋 Steps in this Notebook:
1. **Install dependencies first.**
2. **Locate raw dataset path** across Kaggle/Colab candidates.
3. **Audit data quality:** Scan corrupted images, exact duplicate hashes, train-test leakage, and cross-class label conflicts.
4. **Build Clean Dataset:**
   - Normalize class names (`_test` removal, lowercase).
   - Remove exact train-test leakage images.
   - Remove cross-class contradictory label images.
   - Deduplicate identical copies within classes (keeping 1 unique image per hash).
5. **Stratified Split:** 80% Train, 10% Validation, 10% Test.
6. **EDA on Clean Dataset:** Distribution tables, bar plots, and clean visual sample gallery.
7. **Export Clean Metadata:** Save `clean_dataset.csv` for model development.

In [ ]:
# Step 1: Install all required dependencies first
!pip install -q tensorflow numpy pandas matplotlib seaborn scikit-learn pillow opencv-python
print('✅ Dependencies installed successfully!')


In [ ]:
# Step 2: Import required libraries
import os
import hashlib
from collections import defaultdict, Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
print('✅ Libraries imported successfully!')


In [ ]:
# Step 3: Locate Dataset Path Automatically
def find_dataset_root():
    candidates = [
        '/kaggle/input/datasets/kushagra3204/wheat-plant-diseases/data',
        '/kaggle/input/wheat-plant-diseases/data',
        '/kaggle/input/wheat-plant-diseases',
        '/kaggle/input/data',
        './data'
    ]
    for p in candidates:
        if os.path.exists(p) and os.path.exists(os.path.join(p, 'train')):
            return p
    for root, dirs, files in os.walk('/kaggle/input' if os.path.exists('/kaggle/input') else '.'):
        if 'train' in dirs:
            return root
    return None

DATASET_PATH = find_dataset_root()
TRAIN_PATH = os.path.join(DATASET_PATH, 'train')
TEST_PATH = os.path.join(DATASET_PATH, 'test') if os.path.exists(os.path.join(DATASET_PATH, 'test')) else None

print(f'📁 Dataset Root: {DATASET_PATH}')
print(f'📁 Train Path:   {TRAIN_PATH}')
print(f'📁 Test Path:    {TEST_PATH}')


In [ ]:
# Step 4: Class Name Normalization Function
def normalize_class_name(class_name):
    """Normalizes folder names so 'Black Rust' and 'black_rust_test' align."""
    name = class_name.lower().strip()
    name = name.replace('_test', '')
    name = name.replace('_', ' ')
    return name

def get_image_hash(file_path):
    """Computes MD5 hash to identify identical image files."""
    with open(file_path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

print('Normalization & hashing functions ready.')


In [ ]:
# Step 5: Scan All Image Files in Raw Dataset
image_files = []
splits_to_check = [('train', TRAIN_PATH)]
if TEST_PATH and os.path.exists(TEST_PATH):
    splits_to_check.append(('test', TEST_PATH))

for split_name, split_path in splits_to_check:
    for class_name in os.listdir(split_path):
        class_dir = os.path.join(split_path, class_name)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            fpath = os.path.join(class_dir, fname)
            if os.path.isfile(fpath):
                image_files.append((fpath, split_name, class_name))

print(f'Total raw images scanned: {len(image_files):,}')


In [ ]:
# Step 6: Hash Grouping & Data Leakage / Conflict Audit
hash_map = defaultdict(list)
for fpath, split_name, class_name in image_files:
    try:
        h = get_image_hash(fpath)
        hash_map[h].append((fpath, split_name, class_name))
    except Exception:
        pass

# 1. Identify Train-Test Leakage
same_label_leak_test_files = set()
cross_class_conflict_files = set()

for h, group in hash_map.items():
    train_items = [x for x in group if x[1] == 'train']
    test_items = [x for x in group if x[1] == 'test']
    
    # Check train-test leakage
    if train_items and test_items:
        tr_classes = {normalize_class_name(x[2]) for x in train_items}
        for p, _, tc in test_items:
            if normalize_class_name(tc) in tr_classes:
                same_label_leak_test_files.add(p)
                
    # Check cross-class label conflicts in train
    if len(train_items) > 1:
        tr_classes = {normalize_class_name(x[2]) for x in train_items}
        if len(tr_classes) > 1:
            for p, _, _ in train_items:
                cross_class_conflict_files.add(p)

print('========== DATA QUALITY AUDIT RESULTS ==========')
print(f'Total Duplicate Groups Found:              {sum(1 for g in hash_map.values() if len(g) > 1):,}')
print(f'Train-Test Leakage Test Images Removed:     {len(same_label_leak_test_files):,}')
print(f'Conflicting Cross-Class Train Images Found: {len(cross_class_conflict_files):,}')


In [ ]:
# Step 7: Build Clean Unique Dataset
clean_records = []
seen_clean_hashes = set()

for fpath, split_name, class_name in image_files:
    # Exclude leaked test files and contradictory label files
    if fpath in same_label_leak_test_files or fpath in cross_class_conflict_files:
        continue
    
    norm_class = normalize_class_name(class_name)
    try:
        h = get_image_hash(fpath)
        # Keep only 1 unique copy per hash
        if h not in seen_clean_hashes:
            seen_clean_hashes.add(h)
            clean_records.append((fpath, norm_class))
    except Exception:
        pass

clean_df = pd.DataFrame(clean_records, columns=['filepath', 'class_name'])
print('========== CLEAN DATASET SUMMARY ==========')
print(f'Total Unique Clean Images: {len(clean_df):,}')
print(f'Total Classes:            {clean_df["class_name"].nunique()}')


In [ ]:
# Step 8: Stratified Split (80% Train, 10% Validation, 10% Test)
train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.20,
    random_state=42,
    stratify=clean_df['class_name']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=42,
    stratify=temp_df['class_name']
)

print('========== CLEAN SPLIT SIZES ==========')
print(f'Clean Train:      {len(train_df):,} images (80%)')
print(f'Clean Validation: {len(val_df):,} images (10%)')
print(f'Clean Test:       {len(test_df):,} images (10%)')

# Save clean splits to CSV
clean_df.to_csv('clean_dataset.csv', index=False)
train_df.to_csv('clean_train.csv', index=False)
val_df.to_csv('clean_val.csv', index=False)
test_df.to_csv('clean_test.csv', index=False)
print('✅ Saved clean_dataset.csv, clean_train.csv, clean_val.csv, clean_test.csv')


In [ ]:
# Step 9: Visualizing Clean Class Distribution
dist_df = pd.DataFrame({
    'Train': train_df['class_name'].value_counts(),
    'Validation': val_df['class_name'].value_counts(),
    'Test': test_df['class_name'].value_counts()
}).fillna(0).astype(int).sort_index()

print('Class Distribution Table:')
display(dist_df)

plt.figure(figsize=(14, 6))
dist_df.plot(kind='barh', stacked=True, figsize=(14, 7), color=['#1f77b4', '#ff7f0e', '#2ca02c'])
plt.title('Clean Wheat Plant Disease Dataset: Class Distribution per Split', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clean Unique Images')
plt.ylabel('Disease Category')
plt.legend(title='Split')
plt.tight_layout()
plt.show()


In [ ]:
# Step 10: Clean Sample Visual Gallery (All Classes)
clean_classes = sorted(clean_df['class_name'].unique())
cols = 5
rows = int(np.ceil(len(clean_classes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 3.8 * rows))
axes = axes.flatten()

for i, cls in enumerate(clean_classes):
    sample_path = clean_df[clean_df['class_name'] == cls]['filepath'].iloc[0]
    img = Image.open(sample_path).convert('RGB')
    axes[i].imshow(img)
    axes[i].set_title(f'{cls}\n({img.size[0]}x{img.size[1]})', fontsize=10, fontweight='bold')
    axes[i].axis('off')

for j in range(len(clean_classes), len(axes)):
    axes[j].axis('off')

plt.suptitle('🌾 Sample Leaves from Cleaned Dataset (All Classes)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


### ✅ Summary of Dataset Cleaning & Analysis:
1. **Leakage Eliminated:** Exact copies that appeared in both train and test sets were purged to guarantee un-leaked evaluation.
2. **Contradictions Removed:** Images with conflicting labels across classes were eliminated to prevent conflicting gradients.
3. **Exact Duplicates Cleaned:** Redundant duplicate copies were consolidated so every image hash is unique.
4. **Stratified Split Created:** Clean 80% Train, 10% Validation, and 10% Test partitions ready.

👉 Now open **`model_development.ipynb`** to train the Baseline CNN and Fine-Tuned CNN on this clean dataset!